# MLP classifier limitation

A classifier with the same widths as an autoencoder encoder (`784 → 256 → 64 → 10`) is trained to predict the class. The five stages below show that its weights and its 64-d features still cannot generate a new image: transposed weights are not an inverse, a decoder fit on frozen features loses detail, and sampling that latent space does not produce a coherent object.

This notebook feeds Figure 3 and Q5. Run this on a Colab **GPU** runtime. The project folder must sit at `MyDrive/dl-sesi07`. Change `DRIVE_ROOT` in the setup cell if your folder name differs. Run the cells from top to bottom. Results are written to `artifacts/` on Drive.


In [ ]:
import os
import sys
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/dl-sesi07"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path(DRIVE_ROOT)
else:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if "google.colab" in sys.modules:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])


## Setup

Seed 69, the shared Fashion-MNIST split, and the eight test images used by every reconstruction figure.


In [ ]:
%matplotlib inline
import time

import torch
import torch.nn.functional as F
from torch.optim import Adam
from tqdm.auto import tqdm

from src.config import CLASS_NAMES, GRID_SAMPLES, LATENT_DIM, SEED, get_device, set_seed
from src.data import fixed_test_images, get_splits, make_loader
from src.io import load_json, save_json, save_shared, save_torch
from src.metrics import reconstruction_scores
from src.models import MLPClassifier, MLPDecoder
from src.plots import show_grid

set_seed(SEED)
DEVICE = get_device()
EPOCHS = 20
DECODER_EPOCHS = 20
print("device", DEVICE)

train_set, _, test_set = get_splits()
test_loader = make_loader(test_set, shuffle=False)
test_images, test_labels = fixed_test_images()
save_shared("test8.pt", {"images": test_images, "labels": test_labels})
print("fixed test labels", [CLASS_NAMES[i] for i in test_labels])


## Stage 1. Train the classifier

Cross-entropy fits `P(y | x)`. The 64-d vector is the ReLU activation immediately before the 10-way layer. The assignment asks for at least 88% test accuracy.


In [ ]:
@torch.no_grad()
def test_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        correct += (model(images).argmax(1) == labels).sum().item()
        total += labels.numel()
    return correct / total

classifier = MLPClassifier().to(DEVICE)
optimizer = Adam(classifier.parameters(), lr=1e-3)
history = {"train_loss": [], "test_accuracy": [], "epoch_seconds": []}

for epoch in range(EPOCHS):
    set_seed(SEED + epoch)
    loader = make_loader(train_set, shuffle=True, seed=SEED + epoch)
    classifier.train()
    total_loss = 0.0
    seen = 0
    start = time.perf_counter()
    for images, labels in tqdm(loader, leave=False):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        loss = F.cross_entropy(classifier(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        seen += images.size(0)
    elapsed = time.perf_counter() - start
    accuracy = test_accuracy(classifier, test_loader)
    history["train_loss"].append(total_loss / seen)
    history["test_accuracy"].append(accuracy)
    history["epoch_seconds"].append(elapsed)
    print(f"epoch {epoch + 1:02d}  loss {total_loss / seen:.4f}  test acc {accuracy:.3f}  {elapsed:.1f}s")

if history["test_accuracy"][-1] < 0.88:
    raise RuntimeError("Test accuracy is below 0.88. Raise EPOCHS and run this cell again.")

save_json("mlp", "history.json", history)
save_torch("mlp", "checkpoint.pt", {"classifier": classifier.state_dict()})


## Stages 2 and 3. Transpose the weights

No extra training. PyTorch stores each `Linear` weight as `(out, in)`, so `y @ W` is the same product as `W.T @ y` in the assignment.

Stage 2 pushes a class one-hot back through all three matrices. Stage 3 pushes a real image's 64-d feature back through the first two only. Both should look like blurry templates, because `W.T` is not `W` inverse and ReLU has already discarded the negative part.


In [ ]:
@torch.no_grad()
def project_one_hot(model):
    eye = torch.eye(10, device=next(model.parameters()).device)
    latent = F.relu(eye @ model.fc3.weight)
    hidden = F.relu(latent @ model.fc2.weight)
    pixels = torch.sigmoid(hidden @ model.fc1.weight)
    return pixels.view(10, 1, 28, 28)

@torch.no_grad()
def project_latent(model, latent):
    hidden = F.relu(latent @ model.fc2.weight)
    pixels = torch.sigmoid(hidden @ model.fc1.weight)
    return pixels.view(-1, 1, 28, 28)

classifier.eval()
class_images = project_one_hot(classifier).cpu()
real_latent = classifier.features(test_images.to(DEVICE))
latent_images = project_latent(classifier, real_latent).cpu()

save_torch("mlp", "class_projections.pt", class_images)
save_torch("mlp", "latent_projections.pt", latent_images)
show_grid(class_images, nrow=10, title="Stage 2. Class one-hot, transposed back to pixels")
show_grid(torch.cat([test_images, latent_images]), nrow=8, title="Stage 3. Originals, then transposed latents")

def transpose_reconstruct(images):
    return project_latent(classifier, classifier.features(images))

transpose_scores = reconstruction_scores(transpose_reconstruct, test_loader, DEVICE)
print("transpose", {key: round(value, 4) for key, value in transpose_scores.items()})


## Stage 4. Frozen latent decoder

The classifier stays frozen. A new decoder `64 → 256 → 784` is trained with MSE to rebuild the image from `z_MLP`. Its test MSE, PSNR, and SSIM are the numbers to put next to the vanilla autoencoder, which has the same latent size but was trained to reconstruct.


In [ ]:
for parameter in classifier.parameters():
    parameter.requires_grad = False
classifier.eval()

decoder = MLPDecoder().to(DEVICE)
decoder_opt = Adam(decoder.parameters(), lr=1e-3)
decoder_loss = []

for epoch in range(DECODER_EPOCHS):
    set_seed(SEED + 100 + epoch)
    loader = make_loader(train_set, shuffle=True, seed=SEED + 100 + epoch)
    decoder.train()
    total = 0.0
    seen = 0
    for images, _ in tqdm(loader, leave=False):
        images = images.to(DEVICE)
        with torch.no_grad():
            latent = classifier.features(images)
        restored = decoder(latent)
        pixel_sum = F.mse_loss(restored, images, reduction="sum")
        loss = pixel_sum / images.numel()
        decoder_opt.zero_grad()
        loss.backward()
        decoder_opt.step()
        total += pixel_sum.item()
        seen += images.numel()
    decoder_loss.append(total / seen)
    print(f"decoder epoch {epoch + 1:02d}  mse {total / seen:.4f}")

history["decoder_loss"] = decoder_loss
save_json("mlp", "history.json", history)
checkpoint = torch.load("artifacts/mlp/checkpoint.pt", map_location=DEVICE, weights_only=False)
checkpoint["decoder"] = decoder.state_dict()
save_torch("mlp", "checkpoint.pt", checkpoint)

decoder.eval()

def frozen_reconstruct(images):
    return decoder(classifier.features(images))

frozen_scores = reconstruction_scores(frozen_reconstruct, test_loader, DEVICE)
frozen_preview = frozen_reconstruct(test_images.to(DEVICE)).cpu()
save_torch("mlp", "frozen_recon.pt", frozen_preview)
show_grid(torch.cat([test_images, frozen_preview]), nrow=8, title="Stage 4. Originals, then frozen decoder")
print("frozen decoder", {key: round(value, 4) for key, value in frozen_scores.items()})
try:
    vanilla = load_json("vanilla_ae", "metrics.json")
    print("vanilla AE mse", round(vanilla["test_mse"], 4))
except FileNotFoundError:
    print("Vanilla AE is not saved yet. Notebook 08 prints the comparison.")


## Stage 5. Sample the classifier latent

Draw `z ~ N(mu, diag(sigma^2))` from the test-set features and decode with the stage-4 decoder. This is the prior the classifier actually has. It is not trained to be a smooth generative manifold, so the samples should not become new, complete garments.


In [ ]:
latents = []
with torch.no_grad():
    for images, _ in test_loader:
        latents.append(classifier.features(images.to(DEVICE)).cpu())
z_test = torch.cat(latents)
mu = z_test.mean(0)
sigma = z_test.std(0).clamp_min(1e-6)
generator = torch.Generator().manual_seed(SEED)
epsilon = torch.randn(GRID_SAMPLES, LATENT_DIM, generator=generator)
samples = decoder((mu + sigma * epsilon).to(DEVICE)).cpu()

save_torch("mlp", "prior_samples.pt", samples)
save_torch("mlp", "z_test.pt", {"mu": mu, "sigma": sigma})
show_grid(samples, nrow=8, title="Stage 5. Samples from the classifier latent")

save_json(
    "mlp",
    "metrics.json",
    {
        "test_accuracy": history["test_accuracy"][-1],
        "sec_per_epoch": sum(history["epoch_seconds"]) / len(history["epoch_seconds"]),
        "transpose_mse": transpose_scores["test_mse"],
        "transpose_psnr": transpose_scores["test_psnr"],
        "transpose_ssim": transpose_scores["test_ssim"],
        "frozen_mse": frozen_scores["test_mse"],
        "frozen_psnr": frozen_scores["test_psnr"],
        "frozen_ssim": frozen_scores["test_ssim"],
    },
)
